In [13]:
import pandas as pd
import numpy as np

In [14]:
url = "https://raw.githubusercontent.com/npradaschnor/Pima-Indians-Diabetes-Dataset/master/diabetes.csv"
columns = ['Pregnancies','Glucose', 'BloodPressure', 'SkinThickness','Insulin', 'BMI','DiabetesPedigreeFunction','Age', 'Outcome']

df = pd.read_csv(url, names=columns, header=0)

In [15]:
cols_with_missing_values = ['Glucose', 'BloodPressure', 'SkinThickness','Insulin', 'BMI']
df[cols_with_missing_values] = df[cols_with_missing_values].replace(0,np.nan)
df.fillna(df.mean(), inplace=True)
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148.0,72.0,35.00000,155.548223,33.6,0.627,50,1
1,1,85.0,66.0,29.00000,155.548223,26.6,0.351,31,0
2,8,183.0,64.0,29.15342,155.548223,23.3,0.672,32,1
3,1,89.0,66.0,23.00000,94.000000,28.1,0.167,21,0
4,0,137.0,40.0,35.00000,168.000000,43.1,2.288,33,1


In [16]:
X = df.iloc[:,:8]
y = df['Outcome']

In [17]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

In [18]:
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=28)
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
y_scores = model.predict_proba(X_test)[:,1]


In [19]:
y_scores

array([0.40172347, 0.12200487, 0.03284596, 0.33268945, 0.04595378,
       0.13105674, 0.31467093, 0.82603139, 0.04178563, 0.25889316,
       0.10533451, 0.90478811, 0.1164707 , 0.62413573, 0.11859964,
       0.28332391, 0.31146392, 0.0449468 , 0.05173732, 0.28773008,
       0.01425084, 0.65927473, 0.26932223, 0.11280248, 0.67772305,
       0.14803221, 0.14955453, 0.08260427, 0.12260767, 0.23398904,
       0.19637842, 0.33691962, 0.63416479, 0.86501324, 0.6552725 ,
       0.06632848, 0.2345293 , 0.48455759, 0.0512655 , 0.662818  ,
       0.26140471, 0.87572761, 0.67915961, 0.16579308, 0.92959111,
       0.82246186, 0.42056644, 0.04147335, 0.93989047, 0.04238171,
       0.16493731, 0.12711624, 0.16620951, 0.07215071, 0.15196704,
       0.2032596 , 0.05667409, 0.3875911 , 0.30859084, 0.41231415,
       0.37836347, 0.18289558, 0.21121764, 0.20793656, 0.23435272,
       0.03945698, 0.10960202, 0.08375654, 0.98950423, 0.01920764,
       0.05287932, 0.06310788, 0.74813769, 0.1014552 , 0.45251

In [20]:
from sklearn.metrics import roc_curve
fpr, tpr,thresholds = roc_curve(y_test, y_scores)

In [21]:
import plotly.graph_objects as go
import numpy as np


# Generate a trace for ROC curve
trace0 = go.Scatter(
    x=fpr,
    y=tpr,
    mode='lines',
    name='ROC curve'
)

# Only label every nth point to avoid cluttering
n = 10  
indices = np.arange(len(thresholds)) % n == 0  # Choose indices where index mod n is 0

trace1 = go.Scatter(
    x=fpr[indices], 
    y=tpr[indices], 
    mode='markers+text', 
    name='Threshold points', 
    text=[f"Thr={thr:.2f}" for thr in thresholds[indices]], 
    textposition='top center'
)


# Diagonal line
trace2 = go.Scatter(
    x=[0, 1], 
    y=[0, 1], 
    mode='lines', 
    name='Random (Area = 0.5)', 
    line=dict(dash='dash')
)

data = [trace0, trace1, trace2]

# Define layout with square aspect ratio
layout = go.Layout(
    title='Receiver Operating Characteristic',
    xaxis=dict(title='False Positive Rate'),
    yaxis=dict(title='True Positive Rate'),
    autosize=False,
    width=800,
    height=800,
    showlegend=False
)

# Define figure and add data
fig = go.Figure(data=data, layout=layout)

# Show figure
fig.show()


In [22]:
optimal_idx = np.argmax(tpr - fpr)
optimal_threshold = thresholds[optimal_idx]
print("Optimal threshold is:", optimal_threshold)

Optimal threshold is: 0.3326894470349365


In [24]:
import numpy as np
import plotly.graph_objects as go
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import roc_curve, roc_auc_score
from sklearn.preprocessing import StandardScaler

# Assuming that X_train, X_test, y_train, y_test are already defined

# SVM requires feature scaling for better performance
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Logistic Regression model
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train, y_train)
lr_scores = lr_model.predict_proba(X_test)[:,1]

# SVM model
svm_model = SVC(probability=True)
svm_model.fit(X_train_scaled, y_train)
svm_scores = svm_model.predict_proba(X_test_scaled)[:,1]

# Generate ROC curve data for logistic regression model
lr_fpr, lr_tpr, lr_thresholds = roc_curve(y_test, lr_scores)
lr_auc = roc_auc_score(y_test, lr_scores)

# Generate ROC curve data for SVM model
svm_fpr, svm_tpr, svm_thresholds = roc_curve(y_test, svm_scores)
svm_auc = roc_auc_score(y_test, svm_scores)

# Generate a trace for the Logistic Regression ROC curve
trace0 = go.Scatter(
    x=lr_fpr,
    y=lr_tpr,
    mode='lines',
    name=f'Logistic Regression (Area = {lr_auc:.2f})'
)

# Generate a trace for the SVM ROC curve
trace1 = go.Scatter(
    x=svm_fpr,
    y=svm_tpr,
    mode='lines',
    name=f'SVM (Area = {svm_auc:.2f})'
)

# Diagonal line
trace2 = go.Scatter(
    x=[0, 1], 
    y=[0, 1], 
    mode='lines', 
    name='Random (Area = 0.5)', 
    line=dict(dash='dash')
)

data = [trace0, trace1, trace2]

# Define layout with square aspect ratio
layout = go.Layout(
    title='Receiver Operating Characteristic',
    xaxis=dict(title='False Positive Rate'),
    yaxis=dict(title='True Positive Rate'),
    autosize=False,
    width=800,
    height=800,
    showlegend=True
)

# Define figure and add data
fig = go.Figure(data=data, layout=layout)

# Show figure
fig.show()

